# Sub-sampling configuration sensitivityHow performance and runtime respond to the pairs-per-anchor `s` and the pair mini-batch size `b`.## About this notebookThe model, loss, sampler, metrics and data generation all live in the`rnn_agt` package. This notebook sets up an experiment and reports it; nothingis redefined here, so every notebook and driver in the repository shares oneimplementation.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
import timefrom rnn_agt.diagnostics import check_subsampling_unbiasednessseeds = make_seeds(2026)rng = seeds.data()tau = D.calibrate_tau(1000, D.f_interaction, "normal", rng, 0.50,                      D.DEPENDENCE_SPECS["ar1"])train_subjects = D.make_dataset(1000, "interaction", "normal", rng,                                dependence="ar1", tau=tau)test_subjects  = D.make_dataset(2000, "interaction", "normal", rng,                                dependence="ar1", tau=tau)print(f"{len(train_subjects)} train subjects, tau={tau:.1f}")

### Unbiasedness holds regardless of `s`Theorem A.2 says the subsampled objective is unbiased for the full one at any `s`; `s` controls variance, not bias. Confirm that before interpreting the sweep, since a biased estimator would make the whole comparison meaningless.

In [ ]:
for s in (2, 5, 10, 30):    chk = check_subsampling_unbiasedness(train_subjects[:120], 3, n_draws=300, s=s)    print(f"s={s:3d}  exact={chk['exact']:.3f}  MC mean={chk['mc_mean']:.3f}  "          f"ratio={chk['ratio']:.4f}  z={chk['z']:+.2f}")

In [ ]:
# Table 4 layout: rows are (s, b); columns are (n_train, epoch checkpoint).# The epoch grid differs by sample size, because a larger training set reaches# the same effective number of gradient steps in fewer passes.S_GRID = [5, 10, 15, 30]B_GRID = [32, 64, 128]N_EPOCHS = {1000: (5, 10), 5000: (2, 3)}rows = []for s in S_GRID:    for b in B_GRID:        row = {"s": s, "b": b}        for n_train, checkpoints in N_EPOCHS.items():            seeds = make_seeds(2026)            rng = seeds.data()            tau = D.calibrate_tau(n_train, D.f_interaction, "normal", rng, 0.50,                                  D.DEPENDENCE_SPECS["ar1"])            tr = D.make_dataset(n_train, "interaction", "normal", rng,                                dependence="ar1", tau=tau)            te = D.make_dataset(2000, "interaction", "normal", rng,                                dependence="ar1", tau=tau)            cfg = TrainConfig(model="rnn_agt", epochs=max(checkpoints),                              pair_sample_s=s, pair_batch_b=b,                              hidden_dim=64, gru_layers=2, lr=3e-4,                              eval_at_epochs=checkpoints)            t0 = time.time()            res = train_model(tr, te, 3, cfg, make_seeds(11))            elapsed = time.time() - t0            # One fit yields every epoch column: the optimizer state at epoch 5            # does not depend on whether training later continues to 10, so            # checkpointing is equivalent to refitting and costs half the time.            for ep in checkpoints:                m = res.checkpoints[ep]                row[f"n{n_train}_E{ep}_C"] = m["test_cindex"]                row[f"n{n_train}_E{ep}_AMSE"] = m["test_amse"]            row[f"n{n_train}_seconds"] = elapsed        rows.append(row)        cols = "  ".join(            f"n{n}/E{ep}:{row[f'n{n}_E{ep}_C']:.3f}"            for n in N_EPOCHS for ep in N_EPOCHS[n]        )        print(f"s={s:3d} b={b:4d}  {cols}", flush=True)subsample = pd.DataFrame(rows)# Emit the Table 4 fragment. `s` is a multirow span over its `b` values, and# only the data columns take the alternating shading, matching the manuscript.from rnn_agt import latextable4 = {(r["s"], r["b"], n, ep): {"cindex": r[f"n{n}_E{ep}_C"],                                    "amse":   r[f"n{n}_E{ep}_AMSE"]}          for r in rows for n in N_EPOCHS for ep in N_EPOCHS[n]}os.makedirs("results", exist_ok=True)latex.write_fragment(    "results/table4_subsampling.tex", "Table 4: sub-sampling sensitivity",    latex.subsampling_table(table4, S_GRID, B_GRID, N_EPOCHS),)print(open("results/table4_subsampling.tex").read())subsample.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))for b in B_GRID:    sub = subsample[subsample.b == b]    axes[0].plot(sub.s, sub["n1000_E10_C"], marker="o", label=f"b={b}")    axes[1].plot(sub.s, sub["n1000_seconds"], marker="o", label=f"b={b}")axes[0].set_xlabel("pairs per anchor, s")axes[0].set_ylabel("test IPCW C-index (n=1000, epoch 10)")axes[1].set_xlabel("pairs per anchor, s")axes[1].set_ylabel("runtime to 10 epochs (s)")for ax in axes:    ax.legend(); ax.grid(alpha=.3)fig.suptitle("Sub-sampling configuration: accuracy against cost")fig.tight_layout()fig.savefig("results/subsampling_sensitivity.png", dpi=150)plt.show()print("Look for the smallest s where the C-index curve flattens; beyond it you")print("are paying runtime for variance reduction that no longer moves the")print("estimate. Compare the epoch 5 and epoch 10 columns too: a configuration")print("that needs more epochs to reach the same C-index is not cheaper overall.")